# Privify — End-to-End Pipeline Demo

This notebook demonstrates the Privify video anonymization pipeline running
end-to-end: **video in → detection → Gaussian blur → anonymized video out**.

**Important — MVP status.** The detector currently uses `yolov8n.pt`
pre-trained on COCO. It detects *person* (COCO class 0) as a **temporary
proxy** for face detection. License plate detection is not available at this
stage — it requires fine-tuning on WIDER FACE + CCPD, which is the next
milestone. The purpose of this notebook is to validate the pipeline
architecture, not the detection accuracy.

### What this notebook does

1. Clones the Privify repository
2. Installs dependencies
3. Downloads a small public test video
4. Runs detection + Gaussian blur frame-by-frame
5. Displays the input/output side by side and offers a download link

## 1. Setup

In [ ]:
import os

REPO_DIR = "/content/privify"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/jenz26/privify.git {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone.")

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is importable from the repo root.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.anonymizer import Anonymizer
from src.detector import Detector
from src.pipeline import process_video

## 2. Download test video

We use a short Creative-Commons-licensed clip of pedestrians walking.
The video is small enough to process in under a minute on CPU and a few
seconds on a T4 GPU.

> **Bring your own video:** replace `VIDEO_URL` below with any direct
> `.mp4` link, or upload a file to `samples/input.mp4` manually.

In [ ]:
import urllib.request

# TODO: replace with a stable direct-download URL to a short .mp4 clip
# of pedestrians or people (Creative Commons / public domain).
# The URL below is a placeholder — update it before running.
VIDEO_URL = "https://github.com/jenz26/privify/raw/main/samples/input.mp4"

INPUT_PATH = Path("samples/input.mp4")
INPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    print(f"Downloading test video to {INPUT_PATH} ...")
    urllib.request.urlretrieve(VIDEO_URL, str(INPUT_PATH))
    print("Done.")
else:
    print(f"{INPUT_PATH} already exists, skipping download.")

## 3. Initialize the pipeline components

In [ ]:
# Note: the detector is currently using the COCO pre-trained model.
# It detects 'person' as a temporary proxy for 'face'.
# License plate detection requires fine-tuning (next milestone).
detector = Detector(model_path="yolov8n.pt", conf_threshold=0.35)
anonymizer = Anonymizer(blur_kernel_size=51)

## 4. Run the pipeline

Processing time depends on the runtime:
- **T4 GPU:** a few seconds for a 10–30 s clip
- **CPU:** up to a couple of minutes

The pipeline reads each frame, runs detection, applies Gaussian blur to
every detected region, and writes the anonymized frame to the output video.

In [ ]:
OUTPUT_PATH = Path("samples/output_anonymized.mp4")

stats = process_video(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    detector=detector,
    anonymizer=anonymizer,
)

print(
    f"Processed {stats.total_frames} frames in {stats.elapsed_seconds:.1f}s "
    f"({stats.fps_processed:.1f} fps)"
)
print(f"Total detections: {stats.total_detections}")

## 5. Visualize the result

We extract a frame from the midpoint of both the original and the
anonymized video and display them side by side.

In [ ]:
import cv2
import matplotlib.pyplot as plt


def _read_mid_frame(video_path: Path):
    """Return the frame at 50% of the video duration."""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f"Could not read mid-frame from {video_path}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


original_frame = _read_mid_frame(INPUT_PATH)
anonymized_frame = _read_mid_frame(OUTPUT_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(original_frame)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(anonymized_frame)
axes[1].set_title("Anonymized")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 6. Download the anonymized video

In [ ]:
try:
    from google.colab import files

    files.download(str(OUTPUT_PATH))
except ImportError:
    print(f"Not running on Colab. Output saved to: {OUTPUT_PATH.resolve()}")

## Next steps

This notebook demonstrates the pipeline architecture with a placeholder
detector. The following milestones will bring it to production quality:

- **Fine-tuning** the detector on WIDER FACE (faces) and CCPD (license
  plates) to replace the COCO-pretrained proxy with purpose-built
  detection heads.
- **Tracking integration** via ByteTrack for temporal coherence across
  frames — eliminating per-frame flickering where a subject is missed
  in one frame and re-detected in the next.
- **Quantitative evaluation** with mAP@0.5, mAP@0.5:0.95, and a custom
  *privacy leakage rate* metric (fraction of frames where at least one
  ground-truth subject is not blurred).
- **Systematic failure analysis** on edge cases: small faces, heavy
  occlusion, non-European plate formats, low-light conditions.